# Sentiment Analysis – IMDB Movie Reviews
**Dataset:** IMDB 50K Movie Reviews (Kaggle)
**Goal:** Classify reviews as Positive or Negative
**Tech Stack:** Python · NLTK · TF-IDF · Scikit-learn

## 1. Install & Import Libraries

In [ ]:
# Run this once if not installed
# !pip install pandas numpy scikit-learn nltk matplotlib seaborn wordcloud joblib streamlit

import pandas as pd
import numpy as np
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
print('All libraries imported successfully!')

## 2. Load Dataset
Download from Kaggle: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
Place `IMDB Dataset.csv` inside the `data/` folder.

In [ ]:
df = pd.read_csv('data/IMDB Dataset.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Check for missing values
print('Missing values:\n', df.isnull().sum())
print('\nSentiment counts:\n', df['sentiment'].value_counts())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
plt.figure(figsize=(6, 4))
df['sentiment'].value_counts().plot(kind='bar', color=['#4CAF50', '#F44336'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Review length distribution
df['review_length'] = df['review'].apply(len)
plt.figure(figsize=(10, 4))
df[df['sentiment']=='positive']['review_length'].plot(kind='hist', bins=50,
    alpha=0.6, color='green', label='Positive')
df[df['sentiment']=='negative']['review_length'].plot(kind='hist', bins=50,
    alpha=0.6, color='red', label='Negative')
plt.title('Review Length Distribution')
plt.xlabel('Character Count')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)           # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)    # keep letters only
    text = text.lower()
    words = text.split()
    words = [ps.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

# Test the function
sample = 'This movie was <b>absolutely amazing!</b> I loved every scene.'
print('Original:', sample)
print('Cleaned: ', clean_text(sample))

In [ ]:
# Apply cleaning to all reviews (takes ~1-2 minutes)
print('Cleaning all reviews...')
df['clean_review'] = df['review'].apply(clean_text)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print('Done!')
df[['review', 'clean_review', 'label']].head(3)

## 5. Word Cloud

In [ ]:
from wordcloud import WordCloud

pos_text = ' '.join(df[df['label'] == 1]['clean_review'])
neg_text = ' '.join(df[df['label'] == 0]['clean_review'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wc_pos = WordCloud(width=800, height=400, background_color='white',
                   colormap='Greens', max_words=100).generate(pos_text)
axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].set_title('Positive Reviews', fontsize=14)
axes[0].axis('off')

wc_neg = WordCloud(width=800, height=400, background_color='white',
                   colormap='Reds', max_words=100).generate(neg_text)
axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].set_title('Negative Reviews', fontsize=14)
axes[1].axis('off')

plt.suptitle('Most Frequent Words by Sentiment', fontsize=16)
plt.tight_layout()
plt.savefig('images/wordclouds.png', dpi=150)
plt.show()

## 6. Train-Test Split + TF-IDF Vectorization

In [ ]:
X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')

In [ ]:
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)
print(f'Feature matrix shape: {X_train_tf.shape}')

## 7. Train and Compare Models

In [ ]:
models = {
    'Naive Bayes':         MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
    'LinearSVC':           LinearSVC(C=1.0, max_iter=2000),
}

results = {}
for name, clf in models.items():
    clf.fit(X_train_tf, y_train)
    preds = clf.predict(X_test_tf)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f'{name:25s} → Accuracy: {acc:.4f}')

In [ ]:
# Model comparison chart
plt.figure(figsize=(8, 4))
bars = plt.bar(results.keys(), [v * 100 for v in results.values()],
               color=['#64B5F6', '#81C784', '#FF8A65'])
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy (%)')
plt.ylim(80, 95)
for bar, val in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.1,
             f'{val*100:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('images/model_comparison.png', dpi=150)
plt.show()

## 8. Best Model – LinearSVC Evaluation

In [ ]:
best_model = models['LinearSVC']
best_preds = best_model.predict(X_test_tf)

print('Classification Report:')
print(classification_report(y_test, best_preds,
                             target_names=['Negative', 'Positive']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix – LinearSVC')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('images/confusion_matrix.png', dpi=150)
plt.show()

## 9. Save Model & Vectorizer

In [ ]:
joblib.dump(best_model, 'model.pkl')
joblib.dump(tfidf,      'tfidf.pkl')
print('model.pkl saved!')
print('tfidf.pkl saved!')
print('\nNow run: streamlit run app.py')